# Train SAM 3 on tobacco and weed masks
Set the paths below, enable training, and run all cells on your GPU machine.
This is one fine-tuning run from your original `sam3.pt`; weights never download automatically.

Existing V2 masks are accepted as labels: **0 background, 1 tobacco, 2 weed**.
Each 8-connected region becomes an instance, including small regions. Touching plants can form
one target, and separate parts of one plant can form multiple targets; these are derived instances,
not individually annotated plants. No additional annotation work is required.

Campaigns 1–5 train, 6 validates, and 7–8 are held out for the second notebook.
Exact duplicate training images keep their first copy in campaign/numeric order (12 duplicates
in the inspected dataset; some duplicate label files differ).
See [README](README.md) for installation. Defaults only define functions and print instructions.

In [1]:
from pathlib import Path
import json
import sys
import numpy as np
from PIL import Image

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'notebooks/reproduction.py').is_file())
sys.path.insert(0, str(ROOT / 'notebooks'))
from reproduction import new_run


In [2]:
RUN_TRAINING = True  # Set True on the GPU machine, then Restart Kernel and Run All.
DATA = ROOT / 'data/tobacco_original/Tobacco Aerial Dataset V2/Tobacco Aerial Dataset'
# CHECKPOINT = ROOT / 'sam3.pt'  # Or Path(r'D:\models\sam3.pt').
CHECKPOINT = Path("C:\\Users\\natsusa_aki\\academic\\sam3\\sam3.pt")
RUN_NAME = 'sam3-simple-training-03'  # Use a fresh name for each run.
EPOCHS = 60
LEARNING_RATE = 1e-5

# One fixed training recipe, not a parameter search.
RESOLUTION = 1008
ACCUMULATION = 8  # Microbatch size is 1.
SEED = 42
CONFIDENCE = 0.5  # Also used for validation and initial result review.


## Read the existing data
Original images are used directly. Training annotations are generated inside the run.
656 campaign-2 patches have more than 200 connected regions in one class. Only these dense
patches are split along their longest side until each crop fits SAM 3's 200-query limit.
All pixels are retained; generated crops are saved inside the run automatically.
Validation and review use whole original patches. Crops can split a plant at a border,
another limitation of these automatically derived instance targets.

In [3]:
def image_pairs(data, campaigns):
    pairs = []
    for campaign in campaigns:
        folder = data / f'Campaign no. {campaign}' / ('Patch Images' if campaign == 2 else 'Patch images')
        images = sorted((folder / 'data').glob('*.png'), key=lambda p: int(p.stem))
        if not images:
            raise FileNotFoundError(f'No images in {folder / "data"}')
        for image in images:
            mask = folder / 'mask' / image.name
            if not mask.is_file():
                raise FileNotFoundError(mask)
            pairs.append((image, mask))
    return pairs


In [4]:
def connected_instances(labels):
    from scipy.ndimage import label
    if labels.ndim != 2 or not np.isin(labels, [0, 1, 2]).all():
        raise ValueError('Expected a 2D mask: 0 background, 1 tobacco, 2 weed.')
    for category in (1, 2):
        components, count = label(labels == category, structure=np.ones((3, 3)))
        for instance in range(1, count + 1):
            yield category, components == instance


def training_crops(rgb, labels, x=0, y=0):
    """Split only dense samples, preserving every pixel and staying within 200 queries/class."""
    from scipy.ndimage import label
    counts = [label(labels == k, structure=np.ones((3, 3)))[1] for k in (1, 2)]
    height, width = labels.shape
    if max(counts) <= 200:
        yield rgb, labels, [x, y, width, height]
    elif width >= height:
        mid = width // 2
        yield from training_crops(rgb[:, :mid], labels[:, :mid], x, y)
        yield from training_crops(rgb[:, mid:], labels[:, mid:], x + mid, y)
    else:
        mid = height // 2
        yield from training_crops(rgb[:mid], labels[:mid], x, y)
        yield from training_crops(rgb[mid:], labels[mid:], x, y + mid)


def prepare_annotations(data, pairs, output):
    """Write SAM 3 COCO input, saving crops only for overly dense samples."""
    import hashlib
    from pycocotools import mask as mask_api
    images, annotations, seen, duplicates = [], [], set(), []
    for image_path, mask_path in pairs:
        rgb = np.asarray(Image.open(image_path).convert('RGB'))
        digest = hashlib.sha256(str(rgb.shape).encode() + rgb.tobytes()).hexdigest()
        if digest in seen:
            duplicates.append(image_path.relative_to(data).as_posix())
            continue  # Deterministic: keep the first image in campaign/numeric order.
        seen.add(digest)
        labels = np.asarray(Image.open(mask_path))
        if labels.shape != rgb.shape[:2]:
            raise ValueError(f'Image/mask dimensions differ: {image_path}')
        for crop_rgb, crop_labels, box in training_crops(rgb, labels):
            image_id = len(images) + 1
            filename = image_path.relative_to(data).as_posix()
            if crop_labels.shape != labels.shape:
                crop_folder = output.parent / 'training_crops'
                crop_folder.mkdir(exist_ok=True)
                crop_path = crop_folder / f'{image_id}.png'
                Image.fromarray(crop_rgb).save(crop_path)
                filename = crop_path.resolve().as_posix()
            images.append({'id': image_id, 'file_name': filename,
                           'width': crop_rgb.shape[1], 'height': crop_rgb.shape[0],
                           'source_file': image_path.relative_to(data).as_posix(),
                           'crop_xywh': box})
            for category, mask in connected_instances(crop_labels):
                rle = mask_api.encode(np.asfortranarray(mask.astype(np.uint8)))
                area, bbox = float(mask_api.area(rle)), mask_api.toBbox(rle).tolist()
                rle['counts'] = rle['counts'].decode('ascii')
                annotations.append({'id': len(annotations) + 1, 'image_id': image_id,
                                    'category_id': category, 'segmentation': rle,
                                    'area': area, 'bbox': bbox, 'iscrowd': 0})
    output.write_text(json.dumps({'images': images, 'annotations': annotations,
        'categories': [{'id': 1, 'name': 'tobacco plant'}, {'id': 2, 'name': 'weed plant'}]}))
    print(f'Unique source images: {len(seen)}; training samples: {len(images)}; duplicates excluded: {len(duplicates)}')
    return duplicates


## SAM 3 training input and losses
Each image is queried for both classes, including absent classes. Use SAM 3’s matcher, box/class losses, and focal plus Dice mask losses. The loop runs in one process for native Windows.

In [5]:
def training_dataset(data, annotations, resolution):
    from functools import partial
    from sam3.train.data.sam3_image_dataset import Sam3ImageDataset
    from sam3.train.data.coco_json_loaders import COCO_FROM_JSON
    from sam3.train.transforms import basic_for_api as T
    from sam3.train.transforms.segmentation import DecodeRle
    transforms = T.ComposeAPI([
        DecodeRle(), T.RandomHorizontalFlip(consistent_transform=True, p=0.5),
        T.ColorJitter(consistent_transform=True, brightness=0.15, contrast=0.15,
                      saturation=0.15, hue=0.025),
        T.RandomResizeAPI(sizes=resolution, consistent_transform=True, square=True),
        T.ToTensorAPI(), T.NormalizeAPI(mean=[0.5]*3, std=[0.5]*3)])
    dataset = Sam3ImageDataset(img_folder=str(data), ann_file=str(annotations),
        transforms=[transforms], max_ann_per_img=200, multiplier=1, training=True,
        load_segmentation=True, use_caching=False,
        coco_json_loader=partial(COCO_FROM_JSON, include_negatives=True, category_chunk_size=1))
    dataset._MAX_RETRIES = 1  # Report bad samples instead of silently replacing them.
    return dataset


In [6]:
def build_criterion():
    import torch
    from sam3.train.matcher import BinaryHungarianMatcherV2, BinaryOneToManyMatcher
    from sam3.train.loss.loss_fns import Boxes, IABCEMdetr, Masks
    from sam3.train.loss.sam3_loss import Sam3LossWrapper
    return Sam3LossWrapper(normalization='local',
        matcher=BinaryHungarianMatcherV2(focal=True,cost_class=2,cost_bbox=5,cost_giou=2,alpha=.25,gamma=2,stable=False),
        o2m_matcher=BinaryOneToManyMatcher(alpha=.3,threshold=.4,topk=4),o2m_weight=2,
        use_o2m_matcher_on_o2m_aux=False,scale_by_find_batch_size=True,
        loss_fns_find=torch.nn.ModuleList([
            Boxes(weight_dict={'loss_bbox':5.,'loss_giou':2.}),
            IABCEMdetr(weak_loss=False,weight_dict={'loss_ce':20.,'presence_loss':20.},
                pos_weight=10,alpha=.25,gamma=2,use_presence=True,pos_focal=False,pad_n_queries=200),
            Masks(weight_dict={'loss_mask':200.,'loss_dice':10.},compute_aux=False,focal_alpha=.25,focal_gamma=2)]))


## Load weights and measure validation IoU
Load original or fine-tuned weights strictly. Validation merges predicted instances into class masks; the higher-confidence mask wins overlaps. Mean IoU excludes background and uses all pixels in campaign 6.

In [7]:
def load_sam3(checkpoint, training=False):
    import torch
    from sam3.model_builder import build_sam3_image_model
    if not torch.cuda.is_available():
        raise RuntimeError('Run enabled SAM 3 cells on the CUDA GPU machine.')
    payload = torch.load(checkpoint, map_location='cpu', weights_only=True)
    state = payload.get('model', payload)
    # Original SAM 3 stores detector.*; our saved weights are already unprefixed.
    if any(k.startswith('detector.') for k in state):
        state = {k.removeprefix('detector.'): v for k, v in state.items()
                 if k.startswith('detector.')}
    model = build_sam3_image_model(device='cpu', eval_mode=not training,
                                  load_from_HF=False, enable_segmentation=True,
                                  enable_inst_interactivity=False, compile=False)
    model.load_state_dict(state, strict=True)
    return model.cuda().train(training)


In [8]:
def predict_classes(processor, image):
    """Predict from RGB and fixed text only; the highest-score mask owns overlaps."""
    state = processor.set_image(image)
    labels = np.zeros((image.height, image.width), dtype=np.uint8)
    confidence = np.zeros(labels.shape, dtype=np.float32)
    for category, prompt in ((1, 'tobacco plant'), (2, 'weed plant')):
        processor.reset_all_prompts(state)
        output = processor.set_text_prompt(prompt=prompt, state=state)
        for mask, score in zip(output['masks'], output['scores']):
            mask = mask.squeeze(0).detach().cpu().numpy().astype(bool)
            score = float(score)
            take = mask & (score > confidence)
            labels[take] = category
            confidence[take] = score
    return labels


In [9]:
def class_iou(intersection, union):
    return np.divide(intersection, union, out=np.full(2, np.nan), where=union > 0)


def validate(model, pairs, resolution, threshold):
    import torch
    from sam3.model.sam3_image_processor import Sam3Processor
    model.eval()
    processor = Sam3Processor(model, resolution=resolution, confidence_threshold=threshold)
    intersection, union = np.zeros(2), np.zeros(2)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
        for image_path, mask_path in pairs:
            prediction = predict_classes(processor, Image.open(image_path).convert('RGB'))
            truth = np.asarray(Image.open(mask_path))
            for i, category in enumerate((1, 2)):
                intersection[i] += np.sum((prediction == category) & (truth == category))
                union[i] += np.sum((prediction == category) | (truth == category))
    values = class_iou(intersection, union)
    return {'tobacco_IoU': float(values[0]), 'weed_IoU': float(values[1]),
            'mean_IoU': float(np.nanmean(values))}


## One training run
Train the heads and last four vision blocks, with a frozen text encoder. Use microbatch 1, accumulation 8, BF16, activation checkpointing, AdamW, warmup/cosine learning rate, and flip/color augmentation.

Each epoch saves `last.pt`; improved validation mean IoU saves `best.pt`. These are inference checkpoints, not optimizer-resume files. The progress bar and epoch time help estimate duration on your GPU; 60 epochs may take a long time. A new run name prevents overwriting an earlier run.

In [10]:
def train(model, dataset, validation_pairs, folder, options):
    import math
    import time
    import torch
    from tqdm.auto import tqdm
    from sam3.train.data.collator import collate_fn_api
    from sam3.model.utils.misc import copy_data_to_device

    for parameter in model.backbone.parameters():
        parameter.requires_grad_(False)
    for block in model.backbone.vision_backbone.trunk.blocks[-4:]:
        for parameter in block.parameters():
            parameter.requires_grad_(True)
    model.backbone.vision_backbone.trunk.use_act_checkpoint = True
    model.use_act_checkpoint_seg_head = True
    head, vision = [], []
    for name, parameter in model.named_parameters():
        if parameter.requires_grad:
            (vision if name.startswith('backbone.') else head).append(parameter)
    optimizer = torch.optim.AdamW([
        {'params': head, 'lr': options['learning_rate']},
        {'params': vision, 'lr': options['learning_rate'] * 0.1}], weight_decay=0.01)
    criterion = build_criterion().cuda()
    history, best = [], -1.0
    for epoch in range(options['epochs']):
        started = time.perf_counter()
        model.train()
        model.backbone.language_backbone.eval()
        dataset.set_epoch(epoch)
        factor = min(1.0, (epoch + 1) / 3) * (1 + math.cos(math.pi * epoch / options['epochs'])) / 2
        for group, scale in zip(optimizer.param_groups, (1.0, 0.1)):
            group['lr'] = options['learning_rate'] * scale * factor
        order = torch.randperm(len(dataset)).tolist()
        optimizer.zero_grad(set_to_none=True)
        total = 0.0
        for step, index in enumerate(tqdm(order, desc=f'Epoch {epoch + 1}/{options["epochs"]}')):
            batch = collate_fn_api([dataset[index]], dict_key='all', with_seg_masks=True)['all']
            batch = copy_data_to_device(batch, 'cuda')
            with torch.autocast('cuda', dtype=torch.bfloat16):
                output = model(batch)
                losses = criterion(output, [model.back_convert(t) for t in batch.find_targets])
                loss = losses['core_loss']
            if not torch.isfinite(loss):
                raise FloatingPointError('Non-finite training loss')
            total += float(loss.detach())
            group_start = step // options['accumulation'] * options['accumulation']
            divisor = min(options['accumulation'], len(order) - group_start)
            (loss / divisor).backward()
            if (step + 1) % options['accumulation'] == 0 or step + 1 == len(order):
                torch.nn.utils.clip_grad_norm_(model.parameters(), 0.1, error_if_nonfinite=True)
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
            del batch, output, losses, loss
        metrics = validate(model, validation_pairs, options['resolution'], options['confidence'])
        if not math.isfinite(metrics['mean_IoU']):
            raise FloatingPointError('Validation did not produce a finite IoU')
        row = {'epoch': epoch + 1, 'loss': total / len(order),
               'seconds': time.perf_counter() - started, **metrics}
        history.append(row)
        payload = {'model': model.state_dict(), 'epoch': epoch + 1,
                   'metrics': metrics, 'options': options}
        torch.save(payload, folder / 'last.pt')
        if metrics['mean_IoU'] > best:
            best = metrics['mean_IoU']
            torch.save(payload, folder / 'best.pt')
        (folder / 'history.json').write_text(json.dumps(history, indent=2))
        print(row)
    return history


In [11]:
if RUN_TRAINING:
    import random
    import platform
    import importlib.metadata
    import torch
    if EPOCHS < 1 or LEARNING_RATE <= 0 or ACCUMULATION < 1:
        raise ValueError('Epochs, learning rate, and accumulation must be positive.')
    if not CHECKPOINT.is_file():
        raise FileNotFoundError(f'Set CHECKPOINT to your original sam3.pt: {CHECKPOINT}')
    if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
        raise RuntimeError('This recipe needs a CUDA GPU with BF16 support.')
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    training_pairs = image_pairs(DATA, range(1, 6))
    validation_pairs = image_pairs(DATA, [6])
    run = new_run(ROOT, RUN_NAME)
    options = {'epochs': EPOCHS, 'learning_rate': LEARNING_RATE, 'seed': SEED,
               'resolution': RESOLUTION, 'accumulation': ACCUMULATION,
               'confidence': CONFIDENCE, 'train_campaigns': [1, 2, 3, 4, 5],
               'validation_campaigns': [6], 'test_campaigns': [7, 8],
               'pretrained_checkpoint': str(CHECKPOINT), 'data': str(DATA),
               'trainable_vision_blocks': 4, 'weight_decay': 0.01,
               'targets': '8-connected regions; touching plants may be grouped'}
    (run / 'config.json').write_text(json.dumps(options, indent=2))
    environment = {'python': sys.version, 'platform': platform.platform(),
                   'torch': str(torch.__version__), 'cuda': torch.version.cuda,
                   'gpu': torch.cuda.get_device_name(),
                   'packages': {d.metadata['Name']: d.version for d in importlib.metadata.distributions()}}
    (run / 'environment.json').write_text(json.dumps(environment, indent=2))
    duplicates = prepare_annotations(DATA, training_pairs, run / 'train.json')
    (run / 'excluded_duplicates.json').write_text(json.dumps(duplicates, indent=2))
    dataset = training_dataset(DATA, run / 'train.json', RESOLUTION)
    model = load_sam3(CHECKPOINT, training=True)
    history = train(model, dataset, validation_pairs, run, options)
    print(f'Saved trained model: {run / "best.pt"}')
else:
    print('Training disabled. Set paths and RUN_TRAINING=True on the GPU machine.')


Unique source images: 2148; training samples: 2856; duplicates excluded: 12


c:\Users\natsusa_aki\academic\tobacco-article\.venv-sam3\Lib\site-packages\timm\models\layers\__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Raw dataset length = 5712


RuntimeError: Error(s) in loading state_dict for Sam3Image:
	Unexpected key(s) in state_dict: "backbone.vision_backbone.sam2_convs.0.dconv_2x2_0.weight", "backbone.vision_backbone.sam2_convs.0.dconv_2x2_0.bias", "backbone.vision_backbone.sam2_convs.0.dconv_2x2_1.weight", "backbone.vision_backbone.sam2_convs.0.dconv_2x2_1.bias", "backbone.vision_backbone.sam2_convs.0.conv_1x1.weight", "backbone.vision_backbone.sam2_convs.0.conv_1x1.bias", "backbone.vision_backbone.sam2_convs.0.conv_3x3.weight", "backbone.vision_backbone.sam2_convs.0.conv_3x3.bias", "backbone.vision_backbone.sam2_convs.1.dconv_2x2.weight", "backbone.vision_backbone.sam2_convs.1.dconv_2x2.bias", "backbone.vision_backbone.sam2_convs.1.conv_1x1.weight", "backbone.vision_backbone.sam2_convs.1.conv_1x1.bias", "backbone.vision_backbone.sam2_convs.1.conv_3x3.weight", "backbone.vision_backbone.sam2_convs.1.conv_3x3.bias", "backbone.vision_backbone.sam2_convs.2.conv_1x1.weight", "backbone.vision_backbone.sam2_convs.2.conv_1x1.bia

In [ ]:
if RUN_TRAINING:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot([r['epoch'] for r in history], [r['loss'] for r in history])
    axes[0].set(title='Training loss', xlabel='Epoch')
    for key in ('tobacco_IoU', 'weed_IoU'):
        axes[1].plot([r['epoch'] for r in history], [r[key] for r in history], label=key)
    axes[1].set(title='Validation IoU', xlabel='Epoch', ylim=(0, 1))
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(run / 'training.png', dpi=150)
    plt.show()
